# When potassium clearance slows: astrocytes, a potassium budget and what one trace cannot tell

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/07_astrocytes_and_potassium.ipynb)

*In Google Colab, the first code cell clones the public repository and puts its package on the path; Colab's own NumPy, Matplotlib and ipywidgets are used, not the locked versions.*
**Question:** If the same neuronal activity releases the same potassium, how much higher and how
much longer does extracellular potassium rise when the assumed clearance is slower, and what could a
measurement of that potassium tell us about astrocytes?
**Evidence:** A synthetic one-compartment construction in seconds (Tables 1–2, Figure 1 and an
interactive explorer). No astrocyte recording, glioma experiment or cognitive prediction is involved.

**Why this matters.** Every action potential ends with potassium (K⁺) leaving the neuron, into an
extracellular space that is only a small fraction of the tissue. If that potassium is not cleared,
it accumulates, and raised extracellular potassium depolarizes neurons and can make them
hyperexcitable ([Djukic et al. 2007](https://doi.org/10.1523/JNEUROSCI.0723-07.2007)). Astrocytes are central to this clearance
([Kofuji & Newman 2004](https://doi.org/10.1016/j.neuroscience.2004.06.008); [Sibille et al. 2014](https://doi.org/10.1113/jphysiol.2013.261735)). At the glioma–brain interface,
astrocytes become reactive, tumour cells take their place, and recent work shows that glioma cells can
both *weaken* potassium buffering ([Curry et al. 2023](https://doi.org/10.1016/j.neuron.2023.01.013)) and *add* potassium to the
extracellular space ([Zhang et al. 2025](https://doi.org/10.1016/j.neuron.2024.10.016)). Seizures are common in patients with glioma, and
potassium dysregulation is one proposed contributor. This notebook builds the smallest potassium budget
in which release and clearance can be separated, and asks what it implies and what a potassium
measurement alone could reveal.

**Aims.** After working through this notebook you should be able to
1. explain, with references, how extracellular potassium is released and cleared, which cells and
   transport mechanisms are involved, and how gliomas are thought to alter this balance;
2. write down a one-compartment potassium budget and say what each term and unit means;
3. solve it by hand and read off its landmarks: plateau, time constant, peak and half-life;
4. convert a potassium change into an ideal equilibrium-potential shift, and say what that number is
   *not*;
5. explain why a potassium trace alone cannot attribute clearance to astrocytes or distinguish more
   release from less clearance, and which measurements would.

**Two tracks.** Most sections have a **clinical and biological track**, which connects the model to
neurophysiology, glioma biology and clinical observation, and a **modeling track**, which gives the
equations, derivations and checks. Read either or both; the code cells and figures are shared.
Acronyms and terms from both fields are explained in the glossary after the next section, and all
references are listed, with links, at the end.

Time is in **seconds**, in contrast to the day-scale growth example in notebook 06. None of our rates
were extracted from the cited studies. Normal or reactive astrocytes and malignant cells with an
astrocyte-like transcriptional program are distinct entities; the model contains neither population
explicitly.

**Place in the perspective:** This is the worked example of AI-assisted development (notebook 04
shows its record and independent check). Notebook 08 continues from ion concentrations to
extracellular potentials and fields.

## AI contribution and verification

**Question.** Neurons release potassium (K⁺) into a narrow extracellular space every time they
repolarize. Astrocytes take much of it up and redistribute it, which keeps extracellular K near its
resting level. At the glioma–brain interface, astrocytes are reactive and the tissue is remodelled, so
it is reasonable to ask what a change in clearance would do. This notebook asks three narrow questions
in a model where the answers can be computed exactly:
- *Magnitude and duration:* for identical release, how do the peak and the time course of excess K
  depend on the assumed clearance rate?
- *Voltage consequence:* how large is the corresponding shift of the ideal K equilibrium potential
  E_K, and what is that quantity *not*?
- *Inference:* if only the extracellular K trace were measured, could one tell astrocytic from other
  clearance?

**Clinical and biological track.** Read the three questions as: if astrocytes near a tumour clear
potassium less well, how much higher and for how long does potassium rise after a burst of activity;
how much would that push neurons toward firing; and could a potassium measurement, for example with an
ion-sensitive electrode in a tissue slice, show that the astrocytes are to blame?

**Modeling track.** The model is a single linear ODE with piecewise-constant input. The three questions
become: the dependence of the plateau $q/\lambda$ and time constant $1/\lambda$ on the clearance rate;
the logarithmic Nernst map from concentration to equilibrium potential; and the structural
non-identifiability of the split $\lambda_a$ versus $\lambda_o$, since only their sum enters.

**AI contribution.** Three kinds of computation are involved, and they need different checks.
- *Trained imaging AI:* none in this notebook.
- *Generative research assistance* (development only; records in `docs/ai-use/`): Codex desktop
  (6 October 2026; model snapshot not captured) helped formulate and implement the excess-K model, the
  ideal Nernst calculation, the controlled comparison, its checks and shared figures; this is the
  repository's worked case ([record](../docs/worked-ai-potassium.md)). On 10 October 2026 a Cursor
  agent (model and settings not captured) wrote an expanded version: the closed-form landmarks and
  Nernst slope in `glioma_ai.neuroglial`, the explorer in `glioma_ai.neuroglial_interactive`, their
  tests, and the explanatory text. On 13 October 2026 a Cursor agent (model and settings not captured)
  restructured the text into two tracks and added the glossary and references, including recent
  (2023–2026) studies and papers by Bergen glioma researchers and collaborators, on request; every
  reference was checked against Crossref and its PubMed abstract.
- *Conventional code* (rerun every time, no model): the exact per-interval solution, the Nernst
  relation and the numerical checks. Rebuilding the notebook calls no language model.

**Design choices awaiting scientific review.** The project asked for astrocytes at the glioma–brain
interface in a clinically grounded perspective, presented didactically with interactive
demonstrations. Within that scope, the following are assistant choices, with their reasons:
- *One well-mixed compartment of fixed volume:* the smallest model in which release and clearance can
  be separated and solved exactly. Spatial buffering, diffusion and the extracellular-volume changes
  that accompany activity are omitted.
- *Linear clearance* (removal proportional to excess K): gives a closed-form solution and a single
  time constant. Transporter- and channel-mediated uptake need not be linear over all concentrations,
  and different mechanisms dominate at different times ([Larsen et al. 2014](https://doi.org/10.1002/glia.22629)).
- *Release of 0.6 mM/s in two 4 s pulses:* chosen so that the reference excess stays below 1 mM while
  reduced clearance produces a clearly larger one. It is not a measured release rate.
- *Clearance 0.8/s (astrocytic) + 0.1/s (other) as the reference,* reduced to 0.2/s or 0/s: time
  constants of about 1.1, 3.3 and 10 s, which make the difference visible within a 24 s window.
- *Baseline extracellular K 3 mM, intracellular K 140 mM, 37 °C:* conventional order-of-magnitude
  values for an ideal Nernst calculation, held fixed.
- *The selection of references and the glossary entries.*

**Verification.**
- *Against an independent solution:* with constant input the package's stepped solution must equal
  the closed form $k(t)=q/\lambda+(k_0-q/\lambda)e^{-\lambda t}$ to rounding error, and the peak at the
  end of a pulse must equal $(q/\lambda)(1-e^{-\lambda T})$ (Table 1).
- *Conservation:* released K = K still extracellular + K cleared by astrocytes + K cleared otherwise,
  at every time step, to about 1e-13 mM.
- *Limits:* with zero clearance K accumulates as $qT$; without release it decays as $e^{-\lambda t}$.
- *Identifiability:* swapping the astrocytic and other shares at the same total leaves the trace
  unchanged (section 7 and the explorer).
- *Nernst:* sign, the 37 °C slope RT/F ≈ 26.7 mV, and the derivative ≈ 8.9 mV per mM at 3 mM
  (tested against a numerical derivative; Table 2).
- *Not verified here:* any physiological rate, release amplitude or compartment volume; agreement
  with recordings; spatial effects; any link to seizures, cognition or tumour growth.

**Evidence status.**
- *Tables 1–2, Figure 1 and the explorer:* synthetic constructions in stated units. They show what
  these equations imply, not what happens in tissue.
- *Development record:* an actual contribution, summarized retrospectively and pending human
  scientific review. Comparative benefit of the assistance is unmeasured.

These roles describe development and provenance. [Assistance records](../docs/ai-workflow.md) retain
the review status; record completeness is separate from scientific quality.

## Glossary: acronyms and key terms

Refer back to these tables as needed. Terms are grouped by field; within each group they are roughly
in the order in which they appear.

**Clinical, biological and physiological terms**

| Term | Meaning |
| --- | --- |
| K⁺, [K⁺]$_o$, [K⁺]$_i$ | Potassium ion; its extracellular and intracellular concentrations, in mM. Resting [K⁺]$_o$ is a few mM; [K⁺]$_i$ is over 100 mM. |
| Extracellular space (ECS) | The narrow fluid-filled space between brain cells, a small fraction of tissue volume, so that released ions change its concentration quickly. |
| Action potential, repolarization | The brief electrical impulse of a neuron; during repolarization K⁺ flows out of the cell. |
| Astrocyte | A star-shaped glial cell that, among many functions, takes up K⁺ and neurotransmitters and couples to other astrocytes through gap junctions. |
| Kir4.1 | An inwardly rectifying K⁺ channel of astrocytes (gene *KCNJ10*), central to K⁺ uptake and spatial buffering ([Djukic et al. 2007](https://doi.org/10.1523/JNEUROSCI.0723-07.2007)). |
| Na⁺/K⁺-ATPase | The sodium–potassium pump, which uses energy to move K⁺ into cells; inhibiting it reduced post-stimulus K⁺ clearance in hippocampal slices ([Larsen et al. 2014](https://doi.org/10.1002/glia.22629)). |
| NKCC1 | A Na⁺/K⁺/2Cl⁻ cotransporter that can move K⁺ into astrocytes and causes them to swell. |
| Spatial buffering | Redistribution of K⁺ by the coupled astrocyte network from regions of high to low [K⁺]$_o$ ([Kofuji & Newman 2004](https://doi.org/10.1016/j.neuroscience.2004.06.008)). |
| Gap junction, connexin43 (Cx43) | Channels that directly connect neighbouring cells; Cx43 is the main astrocytic connexin, and glioma cells can couple to astrocytes through it ([Gui et al. 2025](https://doi.org/10.1093/neuonc/noae237)). |
| Tripartite synapse | A synapse together with the astrocyte process that encloses it ([Sibille et al. 2014](https://doi.org/10.1113/jphysiol.2013.261735)). |
| Conditional knockout (cKO) | Deletion of a gene only in a chosen cell type, for example Kir4.1 only in astrocytes. |
| Ion-sensitive microelectrode | An electrode that measures the activity of one ion, such as K⁺, in tissue ([Heinemann & Lux 1977](https://doi.org/10.1016/0006-8993(77)90903-9)). |
| Reactive astrocytes | Astrocytes remodelled morphologically, molecularly and functionally by injury or disease; a spectrum rather than fixed good or bad types ([Escartin et al. 2021](https://doi.org/10.1038/s41593-020-00783-4)). |
| TAA, GFAP | Tumour-associated astrocytes; glial fibrillary acidic protein, a common astrocyte marker. |
| Excitability, epileptiform activity, seizure | How readily neurons fire; abnormal synchronous discharges; the clinical event they can cause. |
| Spreading depolarization (SD) | A slowly propagating wave of near-complete depolarization of neurons and glia, with a large rise in [K⁺]$_o$ ([Curry et al. 2023](https://doi.org/10.1016/j.neuron.2023.01.013)). |
| ECoG | Electrocorticography: recording from the brain surface, for example during surgery. |
| OPC-like state | One of the malignant cell states of glioblastoma, resembling oligodendrocyte-progenitor cells (notebook 05). |
| Voltage-gated K⁺ channel (Kv) | A K⁺ channel opened by depolarization; Kv4.2 (*KCND2*) and EAG2 are examples expressed by glioma cells ([Zhang et al. 2025](https://doi.org/10.1016/j.neuron.2024.10.016); [Dong et al. 2023](https://doi.org/10.1038/s43018-023-00626-8)). |
| Tumour microtubes, TNTs | Long protrusions that connect glioma cells (notebook 06); tunnelling nanotubes (TNTs) are thinner connections through which, for example, mitochondria can move. |
| Mitochondria transfer | Movement of mitochondria between cells, for example from astrocytes to glioblastoma cells ([Watson et al. 2023](https://doi.org/10.1038/s43018-023-00556-5)). |
| Senescence | A state of permanent cell-cycle arrest with a secretory phenotype, which radiotherapy can induce in astrocytes ([Ji et al. 2024](https://doi.org/10.1002/advs.202304609)). |

**Modeling terms**

| Term | Meaning |
| --- | --- |
| Compartment, well-mixed | A volume treated as having one concentration everywhere. |
| Excess K, $k$ | Extracellular K above the 3 mM reference, in mM. |
| Release $q(t)$ | Prescribed input of K into the compartment, in mM/s, constant within each time interval. |
| Clearance rate λ, $\lambda_a$, $\lambda_o$ | Fraction of the excess removed per second (1/s), split into an astrocytic and an "other" part. |
| Linear (first-order) kinetics | Removal proportional to the amount present, as in radioactive decay. |
| ODE | Ordinary differential equation: an equation for how a quantity changes in time. |
| Closed-form solution | An exact formula for the solution, which can be used to check numerical code. |
| Plateau $q/\lambda$ | The level at which release and clearance balance during a long pulse. |
| Time constant $1/\lambda$, half-life $\ln 2/\lambda$ | Time to cover 63% of the way to the plateau; time to decay to half after release stops. |
| Exact interval update | Advancing the solution over each time interval with the closed form, so the step size introduces no error for piecewise-constant input. |
| Mass balance (conservation) | Released K = retained + cleared, checked at every step. |
| Nernst equation, E_K | The ideal equilibrium potential of a K-selective membrane, $E_K=(RT/F)\ln([K]_o/[K]_i)$. |
| RT/F | The thermal voltage, about 26.7 mV at 37 °C. |
| Identifiability | Whether parameters can be determined from a given kind of data ([Raue et al. 2009](https://doi.org/10.1093/bioinformatics/btp358)). Here only $\lambda_a+\lambda_o$ is identifiable from a K trace. |
| Electrodiffusion | Ion movement driven by both concentration gradients and electric fields (Nernst–Planck), as in detailed astrocyte–neuron models ([Halnes et al. 2013](https://doi.org/10.1371/journal.pcbi.1003386); [Sætra et al. 2020](https://doi.org/10.1371/journal.pcbi.1007661)). |
| Synthetic construction | Inputs and parameters prescribed by the modeler, not measured. |

## 1. Why extracellular potassium is worth modelling

- **Release.** Each action potential ends with K⁺ leaving the neuron. The extracellular space is a
  small fraction of tissue volume, so repeated firing can raise extracellular K measurably.
- **Clearance.** Astrocytes take up K⁺ through inward-rectifying Kir4.1 channels and the Na⁺/K⁺-ATPase
  and can redistribute it through their coupled network (spatial buffering). Sibille et al. (2014)
  perturbed astroglial Kir4.1 in hippocampal slices to study this clearance.
- **Consequence.** The K gradient across membranes sets the K equilibrium potential E_K. A higher
  extracellular K makes E_K less negative, which can change neuronal excitability.

**Clinical and biological track.**
- *How high does potassium go?* With ion-sensitive microelectrodes in cat cortex, stimulus-induced
  rises of extracellular potassium reached a "ceiling" of about 10 mM during repetitive stimulation
  and self-sustained afterdischarges ([Heinemann & Lux 1977](https://doi.org/10.1016/0006-8993(77)90903-9)), from a resting level of a
  few mM.
- *Who clears it?* Astrocytes are central: they take up K⁺ and the coupled astrocyte network can
  disperse local rises (spatial buffering; review: [Kofuji & Newman 2004](https://doi.org/10.1016/j.neuroscience.2004.06.008)). Astroglial Kir4.1
  carries about 80% of the synaptically evoked astrocyte current in hippocampal slices, and, as shown
  with glial conditional Kir4.1 knockout mice, astroglial potassium uptake reduces synaptic responses to
  repetitive stimulation and post-tetanic potentiation ([Sibille et al. 2014](https://doi.org/10.1113/jphysiol.2013.261735)).
  Mice lacking astrocytic Kir4.1 showed impaired K⁺ and glutamate uptake, ataxia and stress-induced
  seizures ([Djukic et al. 2007](https://doi.org/10.1523/JNEUROSCI.0723-07.2007)). The mechanisms differ in timing: in hippocampal slices,
  inhibiting Na⁺/K⁺-ATPase reduced post-stimulus clearance, Kir4.1 enacted spatial buffering only during
  the local rise, and inhibiting NKCC1 did not affect the removal rate
  ([Larsen et al. 2014](https://doi.org/10.1002/glia.22629)).
- *What changes near a glioma?* Glioma cell lines have Kir currents more than 20-fold smaller than
  mature astrocytes and depolarized membranes ([Olsen & Sontheimer 2004](https://doi.org/10.1002/glia.10346)), so tumour cells
  replacing astrocytes would be poor K⁺ buffers. Tumours from patients with seizures are enriched for
  synapse-formation genes; the tumour protein IGSF3 interacts with Kir4.1 to suppress potassium
  buffering, drives spreading depolarization and glioma progression in mice, and seizure patients show
  reduced expression of potassium handlers in proliferating tumour cells
  ([Curry et al. 2023](https://doi.org/10.1016/j.neuron.2023.01.013)). In epileptic glioblastoma, OPC-like tumour cells at the tumour–neuron
  interface express the K⁺ channel Kv4.2 and raise extracellular K⁺ and neuronal excitability
  ([Zhang et al. 2025](https://doi.org/10.1016/j.neuron.2024.10.016)). Glioma cells also respond to neuronal activity with K⁺ currents
  amplified through gap junctions between tumour cells ([Venkatesh et al. 2019](https://doi.org/10.1038/s41586-019-1563-y)), and an
  EAG2–Kvβ2 K⁺ channel complex at neuron-contacting regions supports glioblastoma growth in mice
  ([Dong et al. 2023](https://doi.org/10.1038/s43018-023-00626-8)). In the terms of this notebook, gliomas may both *reduce clearance* and
  *add release*.
- *Astrocytes near gliomas do much more than buffer potassium.* Reactive astrocytes are remodelled in
  many ways that cannot be reduced to good or bad types ([Escartin et al. 2021](https://doi.org/10.1038/s41593-020-00783-4)).
  Tumour-associated astrocytes help build an immunosuppressive environment
  ([Heiland et al. 2019](https://doi.org/10.1038/s41467-019-10493-6)): glioblastoma-instructed astrocytes induce T-cell death through
  TRAIL ([Faust Akl et al. 2025](https://doi.org/10.1038/s41586-025-08997-x)), and bidirectional astrocyte–tumour ANXA1–FPR1 signalling
  limits tumour-specific immunity ([Andersen et al. 2025](https://doi.org/10.1038/s41586-025-09191-9)). After stroke, tumour-associated
  astrocytes with reduced Ca²⁺ activity appear at the invasive front, and restoring their Ca²⁺
  signalling suppressed stroke-induced glioma progression in models ([Ye et al. 2026](https://doi.org/10.1038/s43018-026-01238-8)).
  Glioma–astrocyte Cx43 gap junctions contribute to temozolomide resistance ([Gui et al. 2025](https://doi.org/10.1093/neuonc/noae237)).
- *Bergen and collaborators.* Astrocytes enhanced glioblastoma growth in co-culture, in a
  contact-dependent manner, and co-injected astrocytes shortened survival in mice; a signature of
  glioblastoma-activated astrocytes was associated with poor prognosis
  ([Mega et al. 2020](https://doi.org/10.1002/glia.23718)). In 45 glioblastomas, the presence of a PDGFRα⁺/GFAP⁺ subset of
  peritumoral astrocytes was associated with shorter overall survival in an exploratory analysis
  ([Leiss et al. 2020](https://doi.org/10.1002/glia.23756)). Tumour-associated glial host cells from xenografts showed a
  stem-like profile and promoted tumour growth ([Leiss et al. 2017](https://doi.org/10.1186/s12885-017-3109-8)). Astrocytes secrete IL-6
  that increases glioma migration and invasion through MMP14 ([Chen et al. 2016](https://doi.org/10.18632/oncotarget.11515); see the
  [2022 correction](https://doi.org/10.18632/oncotarget.28211)). Radiotherapy makes astrocytes senescent, and senescent astrocytes
  promote an immunosuppressive environment and tumour regrowth, while senolytics after radiotherapy
  prolonged survival in preclinical models ([Ji et al. 2024](https://doi.org/10.1002/advs.202304609)). Astrocytes transfer mitochondria
  to glioblastoma cells through GAP43-dependent connections, enhancing proliferation and
  tumorigenicity ([Watson et al. 2023](https://doi.org/10.1038/s43018-023-00556-5); review: [Storevik et al. 2026](https://doi.org/10.1093/neuonc/noag038));
  transfer through tunnelling nanotubes is bidirectional and was seen in vivo
  ([Saenz-de-Santa-Maria et al. 2026](https://doi.org/10.1038/s41467-026-76619-9)), and astrocyte mitochondria can reactivate quiescent
  glioblastoma cells ([Zhu et al. 2026](https://doi.org/10.1038/s41467-026-76828-2)).
- *What the model abstracts.* Of all this, the model keeps only the potassium budget: release in,
  clearance out. "Astrocytic clearance" is a lumped rate, not Kir4.1, the pump or the network.

**Modeling track.** The model keeps the first two items as one balance equation and computes the third
as an ideal equilibrium potential. More detailed models exist at several levels: neuron models with an
interstitial space, ion concentrations, a Na⁺/K⁺ pump and a "glial" uptake function reproduce seizure
discharges and spreading depression when K⁺ is allowed to rise ([Kager et al. 2000](https://doi.org/10.1152/jn.2000.84.1.495));
electrodiffusive models of astrocytes and the extracellular space conserve charge and particles and
show that K⁺ removal is driven by local astrocyte depolarization ([Halnes et al. 2013](https://doi.org/10.1371/journal.pcbi.1003386));
and electrodiffusive neuron models with homeostatic mechanisms show when concentration changes begin to
affect firing ([Sætra et al. 2020](https://doi.org/10.1371/journal.pcbi.1007661)). A one-compartment budget is the limit in which space,
charge and voltage feedback are all removed, which is what makes it exactly solvable and a clean test
of code. Everything else is omitted on purpose and listed in section 2.

## Worked AI-assisted development case (retrospective)

1. **Investigator question:** include astrocytes in a clinically grounded glioma–brain perspective.
2. **Assistant-supported implementation:** a prescribed one-compartment model, exact interval
   solution, Nernst calculation and shared figures.
3. **Units and balance checks:** mM, mM/s and seconds; independent analytic limits and input =
   retained + cleared accounting.
4. **Biological reading:** the assumed clearance changes K excess; a K trace alone cannot identify
   its astrocytic share.
5. **Remaining uncertainty:** rates, compartments and omitted physiology require scientific review
   and measurement. Cognitive effects are not estimated.

[The saved worked-case document](../docs/worked-ai-potassium.md) identifies the original
retrospective record and source. These steps organize existing evidence; they do not reconstruct a
verbatim dialogue, approved parameters or an AI-benefit experiment. Notebook 04 provides the compact
record-to-check demonstration.

## 2. The model in one equation

Let $k$ be the **excess** extracellular K above a fixed 3 mM reference. Then

$$\frac{dk}{dt}=q(t)-(\lambda_a+\lambda_o)\,k .$$

| Symbol | Meaning | Unit |
| --- | --- | --- |
| $k$ | excess extracellular K above 3 mM | mM |
| $q(t)$ | prescribed release, constant within each time interval | mM/s |
| $\lambda_a$ | lumped astrocytic clearance rate | 1/s |
| $\lambda_o$ | lumped clearance by everything else | 1/s |
| $\lambda=\lambda_a+\lambda_o$ | total clearance rate | 1/s |

Read it as a budget: the change of what is in the compartment equals what comes in minus what is
removed. The astrocytic share of removal is $\lambda_a k$; the rest is $\lambda_o k$.

**Omitted on purpose.** The reservoir that receives cleared K is implicit. There is no Kir4.1
conductance, spatial buffering, intracellular storage, rerelease, pump energetics, neuron firing,
tumour density or change in extracellular volume. Cumulative input and clearance share one fixed
reference volume, so they can be added in mM.

**Clinical and biological track.** Think of a small volume of tissue around active synapses. $q$ is how
much potassium active neurons, and in a glioma possibly tumour cells, push into it per second.
$\lambda_a$ is how quickly astrocytes remove it; $\lambda_o$ lumps everything else, such as diffusion
away, neuronal reuptake and, in tumour tissue, uptake by tumour cells. A value of $\lambda=0.9$ per
second means that 90% of the current excess would be removed per second if nothing else changed.

**Modeling track.** A linear, first-order ODE with constant coefficients and piecewise-constant forcing.
Because the equation is linear, solutions superpose: the response to two pulses is the sum of the
responses to each. The two clearance terms are indistinguishable in the dynamics, since only their sum
$\lambda$ appears in $dk/dt$; they differ only in the bookkeeping of where the cleared K goes.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def repository_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)


if IN_COLAB and repository_root() is None:
    checkout = Path("/content/AI-assisted-glioma-research")
    if not checkout.is_dir():
        url = "https://github.com/arvidl/AI-assisted-glioma-research"
        subprocess.run(["git", "clone", "--depth", "1", url, str(checkout)], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(Path.cwd() / "src"))
if repository_root() is None:
    raise RuntimeError("Open this notebook in a checkout of the repository or in Google Colab.")

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, Markdown, display

from glioma_ai.neuroglial import (
    integrate_potassium,
    nernst_slope_mV_per_mM,
    potassium_equilibrium_mV,
    single_pulse_excess_mM,
)
from glioma_ai.neuroglial_examples import astrocyte_example

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
output = root / "outputs/notebook07"
summary = astrocyte_example(output)
summary["parameters"]

## 3. Solving it by hand

While $q$ is constant the equation is linear with constant coefficients, so

$$k(t)=\frac{q}{\lambda}+\Big(k_0-\frac{q}{\lambda}\Big)e^{-\lambda t}.$$

Four landmarks follow directly and explain every curve below:
- **Plateau** $q/\lambda$: the level a long pulse approaches, where release and clearance balance.
- **Time constant** $1/\lambda$: the time to cover 63% of the way to the plateau, or to decay to 37%.
- **Peak of a pulse of length $T$** starting from zero: $(q/\lambda)(1-e^{-\lambda T})$.
- **Half-life of the decay** after release stops: $\ln 2/\lambda$.

Halving the clearance therefore doubles the plateau *and* doubles the time constant: slower clearance
makes excess K both higher and longer-lasting. The package does not use these formulas; it advances
each time interval with the exact interval update. The next cell compares the two.

**Clinical and biological track.** Weaker clearance has two consequences that add up: potassium rises
higher during activity *and* stays raised for longer afterwards. If activity comes in bursts, the
second burst then starts before the first has been cleared, so potassium climbs from burst to burst.

**Modeling track.** Substituting $u=k-q/\lambda$ gives $\dot u=-\lambda u$, so $u$ decays exponentially.
The peak formula is the solution evaluated at $t=T$ with $k_0=0$. For $\lambda T\gg 1$ the peak
approaches the plateau; for $\lambda T\ll 1$ it approaches $qT$, the zero-clearance accumulation, and
becomes nearly independent of λ.

In [ ]:
rows = []
for name, astro in summary["parameters"]["astro_clearance_per_s"].items():
    total = astro + summary["parameters"]["other_clearance_per_s"]
    landmarks = single_pulse_excess_mM(0.6, total, 4.0)
    stepped = integrate_potassium(np.full(400, 0.6), 0.01, astro, 0.1)["excess_mM"][-1]
    assert np.isclose(stepped, landmarks["peak_mM"], rtol=1e-12)
    rows.append(
        f"| {name} | {total:.1f} | {landmarks['time_constant_s']:.2f} | "
        f"{np.log(2) / total:.2f} | {landmarks['plateau_mM']:.2f} | "
        f"{landmarks['peak_mM']:.3f} | {stepped:.3f} |"
    )
display(
    Markdown(
        "**Table 1. Closed-form landmarks for one 4 s pulse of 0.6 mM/s (synthetic).**\n\n"
        "| Scenario | λ (1/s) | 1/λ (s) | Half-life (s) | Plateau q/λ (mM) | "
        "Peak, closed form (mM) | Peak, package (mM) |\n"
        "| --- | --- | --- | --- | --- | --- | --- |\n" + "\n".join(rows)
    )
)

**How to read Table 1.** Each row is one scenario with its total clearance λ (astrocytic plus the
fixed 0.1/s other clearance). The next three columns are the time constant, the half-life and the
plateau from the formulas above. The last two columns compare the closed-form peak of one 4 s pulse
with the package's stepped solution; the cell asserts that they agree to a relative tolerance of 1e-12.

**Interpretation.** Reducing total clearance from 0.9 to 0.3 to 0.1 per second lengthens the time
constant from 1.1 to 3.3 to 10 s (half-lives 0.77, 2.3 and 6.9 s) and raises the plateau from 0.67 to
2.0 to 6.0 mM. The peak rises less, from 0.65 to 1.40 to 1.98 mM, because a 4 s pulse is too short to
reach the higher plateaus.

With the reference clearance the 4 s pulse almost reaches its plateau, because 4 s is several time
constants. With zero astrocytic clearance the plateau is 6 mM, but a 4 s pulse only climbs a third of
the way: the peak is limited by the pulse length rather than by clearance. Peak values alone therefore
mix two things, the clearance and the duration of release.

## 4. Controlled comparison

**Why a controlled comparison.** Table 1 looked at one pulse at a time. Real activity comes in bursts,
and the question that matters is what happens when activity repeats *before* the tissue has fully
recovered. In tissue, many things differ at once between a healthy region and a tumour-infiltrated
one, so a measured difference cannot be pinned on any one of them. In a model we can change exactly
one thing and keep everything else fixed. This section does that: the activity is the same, and only
the assumed astrocytic clearance differs.

**The design.**
- *Release:* two identical pulses of 0.6 mM/s, at [2, 6) and [10, 14) s (shaded in Figure 1), so each
  pulse adds 2.4 mM and the two together 4.8 mM.
- *Astrocytic clearance:* 0.8, 0.2 or 0 per second, labelled "reference", "reduced" and "zero
  astrocytic". "Reference" names a prescribed scenario, not measured healthy physiology.
- *Other clearance:* 0.1 per second in every scenario, so total clearance is 0.9, 0.3 or 0.1 per second.
- *Gap:* the 4 s between the pulses is a design choice. It equals about 3.6 time constants with the
  reference clearance (recovery is almost complete), 1.2 with reduced clearance (partial recovery) and
  0.4 with zero astrocytic clearance (little recovery). One gap therefore shows all three regimes.
- *Window:* 24 s, long enough to watch the decay after the second pulse.

**What the printed values are.** Below the figure, the first line gives the largest excess K over the
24 s window for each scenario (this is the second peak). The second line gives the largest change of
the ideal E_K over the same window, which is reached at the same moment.

**Clinical and biological track.** This is the thought experiment behind an astrocyte knockout, such
as the Kir4.1 deletions of Djukic et al. (2007) and Sibille et al. (2014), or behind a region where
tumour cells have replaced or altered astrocytes: the same neuronal activity, delivered twice, with
progressively weaker astrocytic clearance. The comparison asks two clinical questions in model form:
how much higher does potassium go, and how long does the tissue stay exposed?

**Modeling track.** Everything except $\lambda_a$ is held fixed, so any difference between the curves
is caused by $\lambda_a$ alone. Because $\lambda_o>0$ in all scenarios, even the zero-astrocytic case
has a finite plateau (6 mM) and decays. The comparison is a one-factor sensitivity analysis at three
levels, not a fit; it shows the shape of the dependence on $\lambda_a$, not its value in any tissue.

In [ ]:
display(Image(filename=str(output / "astrocyte_potassium.png")))
print("SYNTHETIC peak excess K (mM):", summary["peak_excess_K_mM"])
print("SYNTHETIC maximum change in K equilibrium (mV):", summary["peak_K_equilibrium_shift_mV"])

**Figure 1. Same release, different clearance (synthetic construction).** Left: excess extracellular
K for three prescribed astrocytic clearance rates (0.8, 0.2 and 0 per second; other clearance 0.1 per
second throughout) under two identical 4 s release pulses (shaded). With slower clearance the excess is
higher, decays more slowly and has not returned to baseline when the second pulse starts, so the second
peak is higher than the first. Right: the corresponding change of the ideal K equilibrium potential
E_K, computed with fixed intracellular K (140 mM) at 37 °C. E_K is an equilibrium value, not the
membrane voltage of any cell.

**How to read the left panel.** The horizontal axis is time in seconds; the vertical axis is the
*excess* extracellular K in mM above the 3 mM reference, so 0 means baseline. Teal is the reference
clearance, orange reduced astrocytic clearance, grey zero astrocytic clearance. Each curve has the same
four phases: a rise during the first pulse (2–6 s), a decay in the gap (6–10 s), a second rise
(10–14 s) and a final decay. Look at three features, each tied to a landmark of section 3:
1. *How high each pulse goes* (peak height; set by the plateau $q/\lambda$ and the pulse length).
2. *How steeply the curve falls after a pulse* (set by the time constant $1/\lambda$ alone).
3. *Where the curve stands when the second pulse begins at 10 s* (what is left over from the first).

**Left panel, curve by curve.**
- *Reference (λ = 0.9/s):* two nearly identical peaks of 0.65 mM. Only 0.02 mM (3% of the first peak)
  remains at 10 s, so the second pulse starts almost from baseline. After the second pulse, the excess
  falls below 0.1 mM within about 2 s.
- *Reduced (λ = 0.3/s):* the first peak is 1.40 mM. 0.42 mM (30% of it) is still present at 10 s, so
  the second peak reaches 1.52 mM, 9% higher than the first. Falling below 0.1 mM takes about 9 s.
- *Zero astrocytic (λ = 0.1/s):* the first peak is 1.98 mM. 1.33 mM (67%) remains at 10 s, so the
  second peak reaches 2.87 mM, 45% higher than the first, and 1.06 mM is still present at the end of the
  window. Falling below 0.1 mM would take about 34 s after the second pulse.

**How to read the right panel.** Same time axis and colours. The vertical axis is the *change* of the
ideal K equilibrium potential E_K, in mV, relative to its baseline of −102.7 mV at 3 mM. Each point is
the left-panel value passed through the Nernst equation of section 6, so the two panels contain the same
information in different units: wherever a left curve rises, the right one rises too, and they peak at
the same moments. The largest shifts are 5.2, 11.0 and 17.9 mV (E_K of −97.5, −91.7 and −84.8 mV).
The *shape* differs: the right curves rise steeply at first and then flatten. Each millimolar of excess
shifts E_K less than the one before, because of the logarithm, so the per-millimolar shift at the
highest point falls from 8.1 mV (reference) to 7.2 and 6.3 mV. For the same reason, the second pulse of
the zero-astrocytic scenario adds 0.89 mM to the peak but only 4.4 mV to the shift (13.5 to 17.9 mV).

**Interpretation.** In this budget, slower astrocytic clearance has three effects that compound:
- *Higher:* each pulse goes higher (larger plateau).
- *Longer:* the excess persists longer (larger time constant). The time-integrated excess, a measure of
  how long and how strongly the tissue is exposed, is 5.3, 15.7 and 37.4 mM·s for the same 4.8 mM
  released, about seven times larger in the zero-astrocytic case than in the reference.
- *Cumulative:* incomplete recovery between pulses makes the second peak higher than the first, so
  potassium builds up from burst to burst.

A less obvious point: in the reduced scenario astrocytes still remove two-thirds of the potassium that
has been cleared by 24 s (3.15 of 4.72 mM; the explorer's right panel shows this split). Weaker clearance
changes mainly *how fast* potassium is cleared, not *who* eventually clears most of it.

The highest total concentration, about 5.9 mM, remains below the roughly 10 mM ceiling measured by
Heinemann & Lux (1977), so the scenarios are not extreme; they are not, however, calibrated to any
tissue. The right panel is an equilibrium potential, not a membrane voltage. It does not show that any
neuron depolarizes by 18 mV or fires (section 6).

## 5. Explore it yourself

**Why explore.** The figure shows three settings. The explorer lets you change release and clearance
and see which landmark of section 3 controls each feature. For each task, first *predict* what will
happen, then move the sliders, then *explain* what you see.

The explorer re-solves the model when a slider is released. Sliders: astrocytic clearance λa and other
clearance λo (1/s), release q (mM/s), pulse width and the gap between the two pulses (s). Three panels:
- **Left:** excess K, the plateau $q/\lambda$ of a long pulse (dashed) and, dotted, the same model
  with the astrocytic and other shares **swapped**.
- **Middle:** the change in ideal E_K.
- **Right:** where the released K went, as stacked areas that always add up to the black line of
  cumulative release. This is the conservation check made visible.

Things to try:
1. Set both clearances to 0. The excess becomes a staircase that never falls: what goes in stays
   (4.8 mM after two 4 s pulses at 0.6 mM/s).
2. Keep λa + λo fixed but move clearance between the two sliders. The dotted and solid curves stay on
   top of each other, while the teal and grey areas on the right trade places.
3. Set the gap to 0 and the width to 10 s (one 20 s pulse), then compare λa = 0.8 and 0.2. Both reach
   their plateaus (0.67 and 2.0 mM), the first within about 3 s, the second only after about 10 s.
4. Halve the release and the clearance together (q = 0.3, λa = 0.4, λo = 0.05). The plateau stays at
   0.67 mM, but the time constant doubles to 2.2 s, so a 4 s pulse no longer reaches it (first peak 0.56 mM; the second, which starts from
   leftover potassium, 0.57 mM).

Moving a slider needs a running kernel. In a static rendering only the initial figure is shown.

**Clinical and biological track.** Task 2 is the key warning: in this model, an extracellular potassium
recording cannot tell whether astrocytes or something else is doing the clearing. Task 4 shows that a
tissue with less activity *and* weaker clearance can reach the same steady level as a normal tissue,
but respond more sluggishly.

**Modeling track.** Task 1 is the λ → 0 limit, $k=qT$ per pulse. Task 4 shows that the plateau depends
on $q/\lambda$ while the dynamics depend on λ alone: steady-state data identify a ratio, time courses
the rate.

In [ ]:
from glioma_ai.neuroglial_interactive import clearance_explorer

clearance_explorer()

## 6. From concentration to voltage: the ideal Nernst relation
**Why convert to voltage.** Neurons respond to voltages, not directly to concentrations. The link
between extracellular potassium and voltage is the K equilibrium potential, the subject of this section.

**The idea.** K⁺ is much more concentrated inside cells than outside. If a membrane let only K⁺ through,
K⁺ would leak out down its concentration gradient. Each ion that leaves carries a positive charge, so
the inside becomes negative, and that negative potential pulls K⁺ back. At one particular voltage the
electrical pull inwards exactly balances the chemical push outwards, and there is no net flow. That
voltage is the **K equilibrium potential**, $E_K$ (also called the K reversal potential). For a
membrane permeable to K⁺ alone it is given by the Nernst equation:
$$E_K=\frac{RT}{F}\ln\frac{[K]_{out}}{[K]_{in}}.$$

| Symbol | Meaning | Value or unit used here |
| --- | --- | --- |
| $E_K$ | equilibrium potential of K⁺, inside relative to outside | mV |
| $R$ | molar gas constant | 8.314 J mol⁻¹ K⁻¹ |
| $T$ | absolute temperature | 310.15 K (37 °C) |
| $F$ | Faraday constant, the charge of one mole of singly charged ions | 96 485 C mol⁻¹ |
| $[K]_{out}$ | extracellular K concentration | 3 mM + excess $k$ |
| $[K]_{in}$ | intracellular K concentration | fixed at 140 mM |
| $\ln$ | natural logarithm | (dimensionless) |

**Reading the equation term by term.**
- $RT/F$ converts a chemical driving force into volts. At 37 °C it is $8.314\times310.15/96\,485\approx
  0.0267$ V, that is, 26.7 mV. The general form is $RT/(zF)$ with the ion's charge number $z$; for K⁺,
  $z=+1$.
- The ratio $[K]_{out}/[K]_{in}$ has no unit, because both concentrations are in mM. Only their
  *ratio* matters: doubling both leaves $E_K$ unchanged.
- *Sign.* Because $[K]_{in}>[K]_{out}$, the ratio is below 1, its logarithm is negative, and so is $E_K$:
  the inside must be negative to hold K⁺ in.
- *Worked example.* At baseline, $\ln(3/140)=-3.843$, so $E_K=26.73\times(-3.843)\approx-102.7$ mV.
  At 6 mM, $\ln(6/140)=-3.150$ and $E_K\approx-84.2$ mV, a shift of +18.5 mV.
- *Rules of thumb.* Doubling $[K]_{out}$ always shifts $E_K$ by $(RT/F)\ln 2\approx 18.5$ mV, whatever
  the starting level, and a tenfold change shifts it by $(RT/F)\ln 10\approx 61.5$ mV. Textbooks often
  write the equation with $\log_{10}$ and the factor 61.5 mV at 37 °C, which is the same thing.
- *Slope.* $dE_K/d[K]_{out}=(RT/F)/[K]_{out}$: about 8.9 mV per mM at 3 mM, but 4.5 mV per mM at 6 mM.
  The same 1 mM rise therefore matters more when the starting level is low.

**Assumptions of the calculation.** Ideal behaviour (concentrations stand in for activities),
intracellular K fixed at 140 mM (in reality it changes as neurons lose K⁺ and astrocytes take it up), charge +1 and
37 °C. It computes an equilibrium voltage, **not the membrane voltage** of a neuron or astrocyte. Other
ions, permeabilities, channel dynamics, spikes and cognitive function are absent.

**Clinical and biological track.** Neurons at rest sit close to, but not at, E_K, because their
membranes are permeable mainly, not only, to K⁺. A less negative E_K therefore tends to depolarize
them, bringing them closer to firing; this is one reason why raised extracellular potassium is linked
to hyperexcitability (Djukic et al. 2007). How much a neuron actually depolarizes depends on its other
conductances, which this notebook does not model.

**Modeling track.** The Nernst equation follows from equating the chemical and electrical
contributions to the electrochemical potential of K⁺. Its derivative $RT/(F[K]_o)$ is what makes equal
concentration steps give diminishing voltage steps. Models that include the membrane potential
explicitly, such as the Goldman–Hodgkin–Katz equation or the electrodiffusive models cited in
section 1, are needed for actual membrane voltages.

In [ ]:
baseline = float(potassium_equilibrium_mV(3.0, 140.0, 37.0))
print(f"Prescribed ideal baseline E_K: {baseline:.2f} mV")
rows = []
for outside in (3.0, 3.5, 4.0, 6.0, 9.0):
    value = float(potassium_equilibrium_mV(outside, 140.0, 37.0))
    rows.append(
        f"| {outside:.1f} | {value:.1f} | {value - baseline:+.1f} | "
        f"{nernst_slope_mV_per_mM(outside):.1f} |"
    )
display(
    Markdown(
        "**Table 2. Ideal E_K for prescribed extracellular K (37 °C, [K]in = 140 mM).**\n\n"
        "| [K]out (mM) | E_K (mV) | Change from 3 mM (mV) | Slope (mV per mM) |\n"
        "| --- | --- | --- | --- |\n" + "\n".join(rows)
    )
)
trace = integrate_potassium(np.full(200, 0.6), 0.05, 0.8, 0.1)
expected = 0.6 / 0.9 * (1 - np.exp(-0.9 * trace["time_s"]))
np.testing.assert_allclose(trace["excess_mM"], expected, atol=2e-15)
balance = trace["excess_mM"] + trace["astro_cleared_mM"] + trace["other_cleared_mM"]
np.testing.assert_allclose(balance, trace["cumulative_release_mM"], atol=2e-14)
print("Input minus retained and cleared K is zero to floating-point precision.")

**How to read Table 2.** Each row is a prescribed extracellular concentration. The second column is
the ideal E_K, the third its change from the 3 mM baseline (−102.7 mV), the fourth the local slope.

**Interpretation.** Raising extracellular K from 3 to 3.5, 4, 6 and 9 mM shifts E_K by +4.1, +7.7,
+18.5 and +29.4 mV, while the slope falls from 8.9 to 3.0 mV per mM. The first millimolar matters
most. The final checks confirm the closed form and the mass balance to floating-point precision.

## 7. Identifiability: the same trace can have different clearance attribution

**Why this matters.** The natural next step after a model like this is to measure extracellular
potassium near a tumour and ask: *are the astrocytes failing?* Before anyone does that experiment, it
is worth asking whether the answer is contained in the measurement at all. A parameter is
**identifiable** if different values of it always produce different data. If two parameter sets
produce exactly the same data, no amount of data, however precise, can tell them apart.

**The argument, step by step.**
1. *Only the sum enters the dynamics.* Written out, $dk/dt=q-\lambda_a k-\lambda_o k=q-(\lambda_a+
   \lambda_o)k$. The trace $k(t)$ is the solution of this equation, so it depends on $\lambda_a$ and
   $\lambda_o$ only through their sum λ.
2. *So any split with the same sum gives the same trace.* With λ = 0.3/s, the reduced scenario of
   Figure 1 ($\lambda_a=0.2$, $\lambda_o=0.1$) gives exactly the same curve as $\lambda_a=0$,
   $\lambda_o=0.3$ (no astrocytic clearance, but more of something else) or $\lambda_a=0.3$,
   $\lambda_o=0$. The orange curve in Figure 1 could equally be labelled "astrocytes absent, other
   clearance tripled".
3. *Converting to voltage does not help.* E_K is a fixed, one-to-one function of $k$ (section 6), so an
   E_K trace contains exactly the same information as the K trace.
4. *What the trace does identify.* After release stops, $k$ decays as $e^{-\lambda t}$, so the decay
   gives λ (for example from the half-life, $\ln 2/\lambda$) without knowing $q$. Given λ, the height
   of the rise gives $q$. A complete time course with known release timing therefore identifies the
   **total** clearance and the release, but never the split.
5. *What a single number identifies.* A steady level alone gives only the ratio $q/\lambda$: twice
   the release with twice the clearance gives the same plateau (explorer task 4). A peak alone mixes
   release, clearance and pulse length (Table 1).

**A correction to intuition.** A high peak is consistent with more release *or* less clearance. A
slow decay is different: in this model it reflects a smaller total clearance and is *not* explained
by more release, because the decay rate does not depend on $q$, provided release has really stopped.
If release continues at a low level after the stimulus, for example from tumour cells, the decay looks
slower than λ alone would make it. Knowing when release stops is therefore part of the measurement.

The next cell runs two splits with the same total λ = 0.9/s ($\lambda_a=0.8$, $\lambda_o=0.1$ and
$\lambda_a=0.2$, $\lambda_o=0.7$) for a single 10 s pulse, asserts that the traces agree and prints how
much each attributes to astrocytes.

In [ ]:
q = np.full(200, 0.6)
a = integrate_potassium(q, 0.05, 0.8, 0.1)
b = integrate_potassium(q, 0.05, 0.2, 0.7)
np.testing.assert_allclose(a["excess_mM"], b["excess_mM"], atol=2e-15)
print(
    "Same K trajectory; final astrocytic removal differs:",
    round(float(a["astro_cleared_mM"][-1]), 3),
    "versus",
    round(float(b["astro_cleared_mM"][-1]), 3),
    "mM-equivalent",
)

**Reading the result.** With constant release of 0.6 mM/s for 10 s, 6.0 mM is released in total. With
$\lambda_a=0.8$, $\lambda_o=0.1$ astrocytes remove 4.74 mM of it; with $\lambda_a=0.2$, $\lambda_o=0.7$
only 1.19 mM. The extracellular traces are identical to rounding error. A conclusion such as
"astrocytes clear 80% of the potassium" would thus be equally supported, and equally unsupported, by a
trace that is in fact produced by astrocytes clearing 20%.

**What would break the tie.** An experiment must add information that the trace lacks. Three kinds of
addition work in this model, each with its own assumption:
- *A selective perturbation.* Record the trace, then block astrocytic uptake and record again. The
  first decay gives $\lambda_a+\lambda_o$, the second gives $\lambda_o$, and the difference is
  $\lambda_a$. This is the logic of the Kir4.1 knockouts of Sibille et al. (2014) and Djukic et al.
  (2007). It assumes the block is selective (it does not change $\lambda_o$ or release) and complete,
  or of known strength. With a partial block of unknown size $f$, the second decay gives
  $\lambda_o+(1-f)\lambda_a$, and the problem reappears with one more unknown. A long-term knockout may
  also lead to compensation by other pathways, which would violate the selectivity assumption.
- *A second observation.* Measure astrocytic uptake directly as well. In the model the cumulative
  astrocytic removal is $\lambda_a\int k\,dt$, and $\int k\,dt$ is known from the trace (5.93 mM·s in
  both runs above). Dividing gives $\lambda_a$: 4.74/5.93 = 0.8 and 1.19/5.93 = 0.2.
- *An independent estimate of release.* This separates $q$ from λ when only peaks or steady levels are
  available, but it does not split λ.

**Clinical and biological track.** Near a glioma, both sides of the budget may change at once: tumour
cells may add potassium (Zhang et al. 2025) and weaken buffering (Curry et al. 2023), while tumour
cells that replace astrocytes buffer poorly (Olsen & Sontheimer 2004). A high potassium peak would be
consistent with any combination of these. A slow recovery would be consistent with weaker clearance by
any route, or with tumour release that continues after neuronal activity stops. Only an experiment that
changes one of them, or measures it directly, can apportion the effect. This matters for treatment
ideas: a hypothetical treatment aimed at restoring astrocytic Kir4.1 could only help where astrocytic clearance is
actually the limiting step, which a potassium trace alone cannot establish.

**Modeling track.** The map from $(\lambda_a,\lambda_o,q)$ to the trace $k(t)$ factors through
$(\lambda_a+\lambda_o, q)$: the split is **structurally** non-identifiable, a property of the model and
the observation, whatever the data quality. This differs from **practical** non-identifiability, where
parameters are identifiable in principle but poorly determined by noisy or limited data. In a fit, a
structural problem shows up as a flat valley in the likelihood along the line $\lambda_a+\lambda_o=$
constant. A profile likelihood ([Raue et al. 2009](https://doi.org/10.1093/bioinformatics/btp358)) would show it as a profile that never rises. With a
single steady-state value, even $q$ and λ are confounded, since only $q/\lambda$ is observed. The two
remedies above correspond to the standard ones: an intervention that changes the model in a known way,
and an additional output. Exercise 3 of the modeling track asks you to write the extended observation
model.

## 8. Glioma relevance

**Why this section.** Sections 2–7 apply to any brain tissue. Here we ask what a glioma could change in
the potassium budget, and how each change would show up in the model. The cited studies suggest three
routes. Each is described with its evidence, its counterpart in the model and what the model can and
cannot say about it. None of the studies measured the clearance rates λ or the release $q$ of this
model in patients, so all numbers below are illustrations from the synthetic construction.

**Route 1: less clearance (smaller $\lambda_a$, possibly a changed $\lambda_o$).**
- *Clinical and biological track.*
  - *Displacement.* Infiltrating tumour cells can come to occupy tissue that astrocytes would otherwise
    cover. Glioma cell
    lines have Kir currents more than 20-fold smaller than mature astrocytes (Olsen & Sontheimer 2004),
    so tumour cells that replace astrocytes would be poor buffers.
  - *Suppression.* The tumour protein IGSF3 interacts with Kir4.1 to suppress potassium buffering, and
    mediated glioma progression and spreading depolarization in mice (Curry et al. 2023).
  - *Remodelling.* Astrocytes near a tumour become reactive. Reactive states are diverse
    (Escartin et al. 2021), so their potassium clearance cannot be assumed to be lower or higher.
    The Bergen studies of tumour-associated and peritumoral astrocytes (Mega et al. 2020;
    Leiss et al. 2020) concerned tumour growth and prognosis, not potassium handling.
  - *Treatment.* Radiotherapy makes astrocytes senescent (Ji et al. 2024). Whether senescent
    astrocytes clear potassium differently is, as far as this notebook's sources go, an open question.
- *In the model:* this is the comparison of Figure 1. Lowering λ from 0.9 to 0.3 per second raises the
  first peak from 0.65 to 1.40 mM and triples the time constant. Tumour-cell uptake belongs to
  $\lambda_o$, and the Olsen data suggest it would be small.

**Route 2: more release (larger $q$).**
- *Clinical and biological track.*
  - *A tumour source of potassium.* In epileptic glioblastoma, OPC-like tumour cells at the tumour–neuron
    interface express Kv4.2 and raise extracellular K⁺ and neuronal excitability
    (Zhang et al. 2025). This adds a release term that is not neuronal and need not follow the timing of
    neuronal activity.
  - *More neuronal activity.* If neurons near the tumour become more excitable and fire more, they
    release more K⁺ per second.
  - *Tumour currents.* Glioma cells respond to neuronal activity with K⁺ currents amplified through gap
    junctions between tumour cells (Venkatesh et al. 2019). The study measured currents, not their net
    effect on extracellular potassium, so in model terms they could belong to release or to clearance.
- *In the model:* release enters linearly, so the peak scales with $q$. A 50% increase in release
  raises the first peak with reference clearance from 0.65 to 0.97 mM. **Combined with route 1 the two
  effects multiply:** with reduced clearance and 50% more release (explorer: λa = 0.2, q = 0.9), the
  first peak is 2.10 mM. That is higher than the zero-astrocytic scenario of Figure 1 (1.98 mM),
  although neither change on its own reaches it.

**Route 3: changed geometry and connections (no counterpart in the model).**
- *Clinical and biological track.* Astrocytes disperse local potassium rises through their coupled
  network (spatial buffering; Kofuji & Newman 2004). Tumour cells form their own gap-junction-coupled
  networks (Venkatesh et al. 2019), and glioma cells also couple to astrocytes through Cx43 gap
  junctions (Gui et al. 2025). Such networks could, in principle, carry potassium away from or towards
  a site, or interrupt the astrocyte network. Changes in extracellular volume would alter how much a
  given release raises the concentration.
- *In the model:* nothing. The model has one compartment with a fixed volume, so it has no distance,
  no network and no volume change.

**Why it could matter clinically.** Seizures are a frequent feature of malignant glioma
(Curry et al. 2023), and two of the mechanisms above come from seizure-associated tumours: IGSF3 was
identified in a screen of genes enriched in tumours from patients with seizures (Curry et al. 2023),
and the Kv4.2-expressing cells were found in epileptic glioblastoma (Zhang et al. 2025). Raised extracellular potassium is one plausible link between a tumour and
hyperexcitability: it makes E_K less negative (section 6), which tends to depolarize neurons
(Djukic et al. 2007). Neurons that fire more release more potassium, which closes a possible positive
feedback loop. The prescribed-release model cannot represent that loop, because $q$ does not depend on
$k$. A further hypothesis, not tested here, links to notebook 06: if raised potassium increases
neuronal activity, it could feed into the activity-dependent tumour growth discussed there. None of
this implies a treatment recommendation.

**Timescales.** Potassium changes over seconds; tumours grow over weeks to months (notebooks 05 and 06).
The model therefore describes the tissue at one moment. As a tumour progresses, $\lambda_a$, $\lambda_o$
and $q$ would drift slowly, and each moment would correspond to a different version of Figure 1.

**Modeling track.**

| Route | Model counterpart | Status in this notebook |
| --- | --- | --- |
| Less clearance | smaller $\lambda_a$ (or $\lambda_o$) | represented; Figure 1 |
| More release | larger $q$, possibly with its own timing | represented by the $q$ slider; prescribed, not mechanistic |
| Geometry and networks | diffusion, coupling, volume | absent |
| Activity feedback | $q=q(k)$ | absent |

- *Combined effects.* The peak is $(q/\lambda)(1-e^{-\lambda T})$: linear in $q$, nonlinear in λ.
  Changes in release and clearance therefore multiply rather than add.
- *Feedback.* Letting release depend on $k$ makes the model nonlinear. With strong enough positive
  feedback, it may have runaway solutions. Models that let potassium act on neuronal firing reproduce
  seizure discharges and spreading depression (Kager et al. 2000).
- *Space.* Route 3 requires a spatial reaction–diffusion or electrodiffusive model (Halnes et al. 2013),
  with more parameters and therefore more demanding identifiability.
- *Timescale separation.* Treating $\lambda_a$, $\lambda_o$ and $q$ as constant over a recording, but
  as slowly varying functions of tumour state over weeks, would connect this model to the growth models
  of notebooks 02, 05 and 06. That connection is proposed, not built.

## 9. What would make this clinically informative?

**Clinical and biological track.**
- Extracellular K⁺ measurements with ion-sensitive electrodes in resected human tissue or slices,
  near and away from the tumour, with the tumour type stated.
- Selective perturbations of astrocytic uptake (for example Kir4.1 or the Na⁺/K⁺-ATPase) and of
  tumour K⁺ channels, to separate clearance from release.
- Simultaneous recordings of neuronal activity (for example ECoG or patch clamp) to estimate release
  independently, and of astrocytic uptake.
- Spatial tissue assays to identify which cells are present at the recording site: astrocytes,
  reactive astrocytes, tumour cells in different states.
- Controlled platforms with mature brain tissue. Rat brain organoids with myelinated neurons, synapses
  and glial cells support real-time study of glioblastoma invasion ([Zhou et al. 2026](https://doi.org/10.1093/neuonc/noaf271)).
  *Proposal, not done by those authors or here:* such a platform could in principle combine
  potassium-sensitive recordings with selective perturbations.

**Modeling track.** Each measurement would need an observation model, for example how an electrode's
reading relates to the compartment concentration $k$, and identifiability would have to be checked
before any fit. A spatial model would be needed to compare near-tumour and distant recordings.

## 10. Summary and exercises

**What this notebook shows.**
- *Slower clearance makes potassium rise higher and last longer.* Both follow from two numbers, the
  plateau $q/\lambda$ and the time constant $1/\lambda$ (Table 1, Figure 1).
- *Repeated activity builds up.* With slow clearance, a second burst starts before the first has been
  cleared, and the second peak is higher (Figure 1).
- *The voltage consequence is logarithmic.* The first millimolar shifts the ideal E_K most (Table 2).
- *A potassium trace cannot attribute clearance.* Only total clearance determines the trace; release
  and clearance are also confounded at steady state (section 7).

**What it does not show.**
- That astrocytes near a glioma clear potassium more slowly, or by how much.
- Any physiological rate, release amplitude or compartment volume.
- Membrane voltage, firing, seizures, spreading depolarization or cognition.
- Any effect on tumour growth, and any of the non-ionic astrocyte–tumour interactions of section 1.

**Exercises: clinical and biological track.**

1. *Interpreting a peak.*
   - *Why:* the peak is often the easiest number to read off a potassium recording, but near a tumour
     the release is unknown.
   - *Task:* in the explorer, set λa = 0.2 (λo stays 0.1) and halve the release to q = 0.3. Read off
     the highest point and compare it with the reference scenario (0.65 mM). Then compare how quickly
     the curve falls after each pulse in the two settings. What does the height tell you, and what
     does the decay tell you?
   - *Hints:* the highest point is about 0.76 mM, close to the reference, although clearance is three
     times slower. Release enters linearly, so halving $q$ halves every point of the curve
     (1.52/2 = 0.76). The decay depends only on λ (section 7, step 4): the half-life is 2.3 s with
     λ = 0.3/s whatever $q$ is, against 0.77 s for the reference. The decay, not the height, carries
     the clearance information, provided release has really stopped.

2. *A slow recovery near a tumour.*
   - *Why:* this is the most likely first observation in a real study, and choosing the next
     experiment means listing the alternatives first.
   - *Task:* a recording near a glioma shows slower potassium recovery after a stimulus than one far
     from it. List at least three biological explanations from sections 1 and 8, and for each an
     experiment that would test it.
   - *Hints:* sort the explanations by model term.
     - Smaller $\lambda_a$: fewer astrocytes (displacement by tumour cells with small Kir currents) or
       suppressed Kir4.1 (IGSF3).
     - Smaller $\lambda_o$: other routes, such as diffusion away from the site, are slower.
     - Release that continues after the stimulus: Kv4.2-expressing tumour cells.

     For the experiments, use the three tie-breakers of section 7 and the list in section 9:
     identify the cells at the recording site, block astrocytic uptake selectively, or measure it
     directly. Also ask whether the distant recording is a fair control. Figure 1 assumed identical
     release, but neuronal activity near and far from a tumour may differ.

3. *Seizures through potassium.*
   - *Why:* seizures are a frequent feature of malignant glioma (Curry et al. 2023), and potassium is
     one candidate link.
   - *Task:* using Curry et al. (2023) and Zhang et al. (2025), explain how a tumour could make
     seizures more likely through potassium, and why this notebook cannot say whether it does in a
     given patient.
   - *Hints:* build the chain in four steps.
     1. Map each study onto a model term: IGSF3 suppressing Kir4.1 means a smaller $\lambda_a$;
        Kv4.2-expressing tumour cells mean a larger $q$.
     2. Use Figure 1 and the combined example of section 8 (peak 2.10 mM) for how high potassium
        could rise, and Table 2 for the E_K shift.
     3. Use section 6 (clinical track) for why a less negative E_K tends to depolarize neurons.
     4. List what is missing: membrane voltage and other conductances, neuronal networks, the
        feedback loop in which more firing releases more potassium, any seizure threshold, and
        measured values of λ and $q$ in that patient.

4. *Beyond potassium.*
   - *Why:* most of the astrocyte–glioma studies cited here, including the Bergen ones, concern
     interactions that do not involve potassium. It is important to see what this model is blind to.
   - *Task:* choose one interaction from section 1 (immunosuppression, mitochondrial transfer,
     senescence after radiotherapy). Describe which measurement would be needed to study it, and why a
     potassium recording would not do.
   - *Hints:* for each interaction, name the quantity that changes in the cited study: T-cell death
     or infiltration (Faust Akl et al. 2025; Andersen et al. 2025), mitochondria moving from
     astrocytes into tumour cells (Watson et al. 2023; Saenz-de-Santa-Maria et al. 2026), or senescent
     astrocytes and tumour regrowth after irradiation (Ji et al. 2024). Then ask whether that
     interaction changes $q$ or λ. If it does not, a potassium trace cannot detect it. Consider also
     the timescale: these processes unfold over much longer times than the seconds modelled here.

5. *From model to experiment (a proposal).*
   - *Why:* section 9 lists ingredients; putting them together shows what an informative experiment
     would cost.
   - *Task:* sketch an experiment on a brain organoid platform such as that of Zhou et al. (2026)
     that could estimate $\lambda_a$ near and away from invading tumour cells.
   - *Hints:* say what you would record (for example extracellular potassium with an ion-sensitive
     electrode), what you would perturb (a selective block of astrocytic uptake, section 7), how you
     would identify the cells at each site, and which controls you need. This is an exercise in
     design; neither those authors nor this notebook did such an experiment.

**Exercises: modeling track.**

1. *Time to plateau.*
   - *Why:* Table 1 showed that a short pulse is limited by its duration rather than by clearance.
     This exercise finds where that changes.
   - *Task:* with section 3's formulas, find the pulse length at which each scenario reaches 90% of
     its plateau. Check the reduced scenario (λ = 0.3/s) in the explorer.
   - *Hints:* set the peak $(q/\lambda)(1-e^{-\lambda T})$ equal to $0.9\,q/\lambda$, so
     $e^{-\lambda T}=0.1$ and $T=\ln 10/\lambda$. This gives 2.6, 7.7 and 23 s for λ = 0.9, 0.3 and
     0.1 per second. In the explorer, set the gap to 0: the two pulses then merge into one of twice the
     width. With λa = 0.2, the curve passes 1.8 mM (90% of 2.0) about 7.7 s after onset. The 23 s for
     λ = 0.1/s is longer than the explorer's longest merged pulse (20 s), which is why the
     zero-astrocytic peak in Figure 1 is set by duration.

2. *Superposition and build-up.*
   - *Why:* superposition explains why repeated bursts accumulate, and it gives every peak without
     simulation.
   - *Task:* derive the second-pulse peak of Figure 1 for λ = 0.3/s and compare it with 1.52 mM. Then
     find the peak that a long train of identical pulses approaches.
   - *Hints:*
     1. First peak: $P_1=(q/\lambda)(1-e^{-\lambda T})$.
     2. Decay over the gap $G$: $k(10)=P_1e^{-\lambda G}$.
     3. Second pulse starting from $k(10)$:
        $P_2=q/\lambda+(k(10)-q/\lambda)e^{-\lambda T}$.

     With $e^{-1.2}=0.301$ this gives 1.398, then 0.421, then $2+(0.421-2)\times0.301=1.52$ mM. For
     the train, apply the same map repeatedly and set $P_{n+1}=P_n$:
     $P_\infty=(q/\lambda)(1-e^{-\lambda T})/(1-e^{-\lambda(T+G)})$. With $T=G=4$ s this gives 0.65,
     1.54 and 3.59 mM. Build-up is negligible with the reference clearance and nearly doubles the first
     peak in the zero-astrocytic case.

3. *Making the split identifiable.*
   - *Why:* section 7's main lesson is that the measurement, not the model, decides what can be
     learned. Here you design the measurement.
   - *Task:* write an extended observation model that makes $\lambda_a$ identifiable, and show that
     it does.
   - *Hints:* take two outputs, $y_1(t)=k(t)$ and $y_2(t)=A(t)=\lambda_a\int_0^t k\,ds$ (cumulative
     astrocytic removal). From $y_1$ alone, λ and $q$ are identified (section 7, step 4) and
     $\int k\,ds$ is known. Then $\lambda_a=y_2/\int k\,ds$ and $\lambda_o=\lambda-\lambda_a$. Check
     with the numbers of section 7: 4.741/5.926 = 0.800. Alternatively, use two experiments with the
     same $q$ and $\lambda_o$, before and after a complete block of astrocytic uptake. Then ask what
     happens if the block removes an unknown fraction $f$ of $\lambda_a$.

4. *Saturating uptake.*
   - *Why:* transporters and pumps have a maximum rate, so linear clearance can only be an
     approximation at low concentrations.
   - *Task:* replace $\lambda_a k$ by $V_{max}k/(K_m+k)$. Which of the checks used here (closed form,
     conservation, limits) still hold? How does the plateau change?
   - *Hints:*
     - *Conservation* holds for any uptake law, provided the cumulative astrocytic removal integrates
       the actual flux.
     - *The closed form* of section 3 is lost.
     - *The low-concentration limit* $k\ll K_m$ recovers the linear model with
       $\lambda_a=V_{max}/K_m$.
     - *Plateau:* solve $q=V_{max}k/(K_m+k)+\lambda_o k$, a quadratic. With $K_m=2$ mM,
       $V_{max}=1.6$ mM/s (so $V_{max}/K_m=0.8$/s), $\lambda_o=0.1$/s and $q=0.6$ mM/s, the plateau is
       0.93 mM instead of the linear 0.67 mM.
     - *No plateau:* if $\lambda_o=0$ and $q>V_{max}$, there is none, and $k$ grows without bound.
     - *Open question:* could saturation make the split identifiable? Compare traces at two release
       amplitudes. In the linear model, doubling $q$ exactly doubles the trace.

## Assistance and acceptance evidence

This is the repository's worked case of AI-assisted development; the record and its independent
check are shown in notebook 04 and in [the worked-case document](../docs/worked-ai-potassium.md). The
closed-form, conservation and Nernst checks are executable evidence about the code. The development
records are retrospective summaries, not invented dialogue or a completed human physiological review.

## References

All links were checked on 13 October 2026; the summaries in this notebook follow the published
abstracts.

**Potassium release, clearance and astrocytes**
- Heinemann U, Lux HD. Ceiling of stimulus induced rises in extracellular potassium concentration in
  the cerebral cortex of cat. *Brain Research* 1977. [doi:10.1016/0006-8993(77)90903-9](https://doi.org/10.1016/0006-8993(77)90903-9)
- Kofuji P, Newman EA. Potassium buffering in the central nervous system. *Neuroscience* 2004.
  [doi:10.1016/j.neuroscience.2004.06.008](https://doi.org/10.1016/j.neuroscience.2004.06.008)
- Djukic B et al. Conditional knock-out of Kir4.1 leads to glial membrane depolarization, inhibition of
  potassium and glutamate uptake, and enhanced short-term synaptic potentiation. *Journal of
  Neuroscience* 2007. [doi:10.1523/JNEUROSCI.0723-07.2007](https://doi.org/10.1523/JNEUROSCI.0723-07.2007)
- Sibille J, Pannasch U, Rouach N. Astroglial potassium clearance contributes to short-term plasticity
  of synaptically evoked currents at the tripartite synapse. *Journal of Physiology* 2014.
  [doi:10.1113/jphysiol.2013.261735](https://doi.org/10.1113/jphysiol.2013.261735)
- Larsen BR et al. Contributions of the Na⁺/K⁺-ATPase, NKCC1, and Kir4.1 to hippocampal K⁺ clearance
  and volume responses. *Glia* 2014. [doi:10.1002/glia.22629](https://doi.org/10.1002/glia.22629)
- Escartin C et al. Reactive astrocyte nomenclature, definitions, and future directions. *Nature
  Neuroscience* 2021. [doi:10.1038/s41593-020-00783-4](https://doi.org/10.1038/s41593-020-00783-4)

**Gliomas and potassium**
- Olsen ML, Sontheimer H. Mislocalization of Kir channels in malignant glia. *Glia* 2004.
  [doi:10.1002/glia.10346](https://doi.org/10.1002/glia.10346)
- Venkatesh HS et al. Electrical and synaptic integration of glioma into neural circuits. *Nature*
  2019. [doi:10.1038/s41586-019-1563-y](https://doi.org/10.1038/s41586-019-1563-y)
- Curry RN et al. Glioma epileptiform activity and progression are driven by IGSF3-mediated potassium
  dysregulation. *Neuron* 2023. [doi:10.1016/j.neuron.2023.01.013](https://doi.org/10.1016/j.neuron.2023.01.013)
- Dong W et al. A designer peptide against the EAG2–Kvβ2 potassium channel targets the interaction of
  cancer cells and neurons to treat glioblastoma. *Nature Cancer* 2023.
  [doi:10.1038/s43018-023-00626-8](https://doi.org/10.1038/s43018-023-00626-8)
- Zhang Y et al. Potassium ion channel modulation at cancer-neural interface enhances neuronal
  excitability in epileptogenic glioblastoma multiforme. *Neuron* 2025.
  [doi:10.1016/j.neuron.2024.10.016](https://doi.org/10.1016/j.neuron.2024.10.016)

**Astrocytes and gliomas (2019–2026)**
- Heiland DH et al. Tumor-associated reactive astrocytes aid the evolution of immunosuppressive
  environment in glioblastoma. *Nature Communications* 2019. [doi:10.1038/s41467-019-10493-6](https://doi.org/10.1038/s41467-019-10493-6)
- Faust Akl C et al. Glioblastoma-instructed astrocytes suppress tumour-specific T cell immunity.
  *Nature* 2025. [doi:10.1038/s41586-025-08997-x](https://doi.org/10.1038/s41586-025-08997-x)
- Andersen BM et al. Barcoded viral tracing identifies immunosuppressive astrocyte–glioma
  interactions. *Nature* 2025. [doi:10.1038/s41586-025-09191-9](https://doi.org/10.1038/s41586-025-09191-9)
- Gui Y et al. Glioma–astrocyte connexin43 confers temozolomide resistance through activation of the
  E2F1/ERCC1 axis. *Neuro-Oncology* 2025. [doi:10.1093/neuonc/noae237](https://doi.org/10.1093/neuonc/noae237)
- Ye Q et al. Stroke drives glioma progression through the emergence of tumor-associated astrocytes
  with reduced Ca²⁺ activity. *Nature Cancer* 2026. [doi:10.1038/s43018-026-01238-8](https://doi.org/10.1038/s43018-026-01238-8)
- Saenz-de-Santa-Maria I et al. Tunneling nanotubes regulate mitochondrial homeostasis between
  glioblastoma and astrocytes, and between tumor cells in vivo. *Nature Communications* 2026.
  [doi:10.1038/s41467-026-76619-9](https://doi.org/10.1038/s41467-026-76619-9)
- Zhu H et al. Metabolic licensing of quiescent glioblastoma activation and immune evasion via
  astrocyte-mitochondria shuttle. *Nature Communications* 2026. [doi:10.1038/s41467-026-76828-2](https://doi.org/10.1038/s41467-026-76828-2)

**Bergen and collaborators**
- Chen W et al. Human astrocytes secrete IL-6 to promote glioma migration and invasion through
  upregulation of cytomembrane MMP14. *Oncotarget* 2016. [doi:10.18632/oncotarget.11515](https://doi.org/10.18632/oncotarget.11515);
  correction 2022: [doi:10.18632/oncotarget.28211](https://doi.org/10.18632/oncotarget.28211)
- Leiss L et al. Tumour-associated glial host cells display a stem-like phenotype with a distinct gene
  expression profile and promote growth of GBM xenografts. *BMC Cancer* 2017.
  [doi:10.1186/s12885-017-3109-8](https://doi.org/10.1186/s12885-017-3109-8)
- Mega A et al. Astrocytes enhance glioblastoma growth. *Glia* 2020. [doi:10.1002/glia.23718](https://doi.org/10.1002/glia.23718)
- Leiss L et al. Platelet-derived growth factor receptor α/glial fibrillary acidic protein expressing
  peritumoral astrocytes associate with shorter median overall survival in glioblastoma patients.
  *Glia* 2020. [doi:10.1002/glia.23756](https://doi.org/10.1002/glia.23756)
- Watson DC et al. GAP43-dependent mitochondria transfer from astrocytes enhances glioblastoma
  tumorigenicity. *Nature Cancer* 2023. [doi:10.1038/s43018-023-00556-5](https://doi.org/10.1038/s43018-023-00556-5)
- Ji J et al. Radiotherapy-induced astrocyte senescence promotes an immunosuppressive
  microenvironment in glioblastoma to facilitate tumor regrowth. *Advanced Science* 2024.
  [doi:10.1002/advs.202304609](https://doi.org/10.1002/advs.202304609)
- Storevik S et al. Mitochondria transfer in glioblastoma. *Neuro-Oncology* 2026.
  [doi:10.1093/neuonc/noag038](https://doi.org/10.1093/neuonc/noag038)
- Zhou W et al. Development of a highly differentiated rat brain organoid model for exploring
  glioblastoma invasion dynamics and therapy. *Neuro-Oncology* 2026. [doi:10.1093/neuonc/noaf271](https://doi.org/10.1093/neuonc/noaf271)

**Modeling**
- Kager H, Wadman WJ, Somjen GG. Simulated seizures and spreading depression in a neuron model
  incorporating interstitial space and ion concentrations. *Journal of Neurophysiology* 2000.
  [doi:10.1152/jn.2000.84.1.495](https://doi.org/10.1152/jn.2000.84.1.495)
- Halnes G et al. Electrodiffusive model for astrocytic and neuronal ion concentration dynamics. *PLoS
  Computational Biology* 2013. [doi:10.1371/journal.pcbi.1003386](https://doi.org/10.1371/journal.pcbi.1003386)
- Sætra MJ, Einevoll GT, Halnes G. An electrodiffusive, ion conserving Pinsky-Rinzel model with
  homeostatic mechanisms. *PLoS Computational Biology* 2020. [doi:10.1371/journal.pcbi.1007661](https://doi.org/10.1371/journal.pcbi.1007661)
- Raue A et al. Structural and practical identifiability analysis of partially observed dynamical
  models by exploiting the profile likelihood. *Bioinformatics* 2009.
  [doi:10.1093/bioinformatics/btp358](https://doi.org/10.1093/bioinformatics/btp358)

**Next:** notebook 08 distinguishes extracellular potential, electric field and cognition; notebook 06
keeps slow growth as a separate model.